In [ ]:
import math
import numpy as np
import torch
import matplotlib.pyplot as plt

import os, sys
os.chdir('..')

from sazz.samplers.AutomaticBoomerangSampler import AutomaticBoomerangSampler
from sazz.samplers.StickyAutomaticBoomerangSampler import StickyAutomaticBoomerangSampler
from sazz.samplers.AutomaticZigZagSampler_torch import AutomaticZigZagSampler
from sazz.samplers.StickyAutomaticZigZagSampler_torch import StickyAutomaticZigZagSampler
from sazz.models.bnn_torch import make_bnn_regression

In [ ]:
def make_dataset_1d(true_f, x_train, x_test_range=(-4.0, 4.0),
                    n_test=300, noise_std_true=0.1, rng=None):
    rng = rng or np.random.default_rng(42)
    y_train = true_f(x_train) + rng.normal(0, noise_std_true, size=x_train.shape)
    x_test = np.linspace(*x_test_range, n_test)
    y_test = true_f(x_test)

    x_mean, x_std = x_train.mean(), x_train.std()
    y_mean, y_std = y_train.mean(), y_train.std()
    x_train_s = (x_train - x_mean) / x_std
    x_test_s  = (x_test  - x_mean) / x_std
    y_train_s = (y_train - y_mean) / y_std
    y_test_s  = (y_test  - y_mean) / y_std

    return {
        "X_train": torch.tensor(x_train_s.reshape(-1, 1), dtype=torch.float64),
        "y_train": torch.tensor(y_train_s, dtype=torch.float64),
        "X_test":  torch.tensor(x_test_s.reshape(-1, 1),  dtype=torch.float64),
        "y_test":  torch.tensor(y_test_s,  dtype=torch.float64),
        "x_mean": x_mean, "x_std": x_std,
        "y_mean": y_mean, "y_std": y_std,
        "x_train_raw": x_train, "y_train_raw": y_train,
        "x_test_raw": x_test,  "y_test_raw": y_test,
        "noise_std_true": noise_std_true,
    }


# --- Three test problems ---
rng = np.random.default_rng(42)
noise_std = 0.05

datasets_1d = {
    "gap": make_dataset_1d(
        true_f=lambda x: np.sin(1.5 * x) + 0.3 * x,
        x_train=np.concatenate([
            rng.uniform(-3.0, -1.5, size=30),
            rng.uniform( 1.5,  3.0, size=30),
        ]),
        noise_std_true=noise_std,
        rng=rng,
    ),
    "sharp": make_dataset_1d(
        true_f=lambda x: 0.3 * np.tanh(x) + 1.5 * np.exp(-((x - 0.5) ** 2) / 0.05),
        x_train=np.concatenate([
            rng.uniform(-3.0, 3.0, size=40),
            rng.uniform( 0.0, 1.0, size=20),
        ]),
        noise_std_true=noise_std,
        rng=rng,
    ),
    "multiscale": make_dataset_1d(
        true_f=lambda x: np.sin(0.5 * x) + 0.3 * np.sin(4.0 * x),
        x_train=rng.uniform(-3.0, 3.0, size=80),
        noise_std_true=noise_std,
        rng=rng,
    ),
}

In [ ]:
target_1d = make_bnn_regression(
    datasets_1d['multiscale']['X_train'],
    datasets_1d['multiscale']['y_train'],
    layer_sizes=[1, 16, 8, 1],     # small enough to compare to HMC
    activation="tanh",
    prior_std_weight=1.0,
    prior_std_bias=1.0,
    fan_in_scaling=True,
    noise_std= noise_std / datasets_1d['multiscale']['y_std'],  # match standardised noise
    covariance_reference="laplace_diag", # hessian_prior_floor
    adam_steps = 3000
)

In [ ]:
from sazz.utils.warmup import find_reference_bnn
x_ref, Sigma_inv = target_1d.x_ref, target_1d.Sigma_inv
eigvals = torch.linalg.eigvalsh(Sigma_inv)
print(f"Sigma_inv eigenvalues: min={eigvals.min():.3e}, max={eigvals.max():.3e}")
print(f"Condition number: {(eigvals.max() / eigvals.min()).item():.3e}")

In [ ]:
from sazz.utils.bnn_utils import make_kappa_from_inclusion, make_kappa_vector_bnn

kappa_from_weights = make_kappa_from_inclusion(layer_sizes=[1, 16, 8, 1], prior_std_weight=1.0,
                                  prior_inclusion_weight=0.5,  # sparsity belief
                                  fan_in_scaling=True)

kappa_boston = make_kappa_vector_bnn(
    target_1d.meta['layer_sizes'],
    #kappa_weights=[0.5, 0.05, 0.5],  # sparse middle layer
    kappa_weights=[2.0, 0.5, 2.0],
    kappa_biases=1e6,
)

In [ ]:
from sazz.utils.bnn_utils import make_kappa_vector_bnn
from sazz.utils.warmup import warmup, tune_refresh_rate

sampler_boston = AutomaticBoomerangSampler(
    grad_target=target_1d.grad_target,
    D=target_1d.D,
    thinning="pli",
    refresh_rate=1.0,
)

sampler_boston.preprocess(
    x_ref=target_1d.x_ref,
    Sigma_inv=target_1d.Sigma_inv
    
)
warmup(sampler_boston, n_rounds=5, n_pilot=5000, target=target_1d)
tune_info = tune_refresh_rate(sampler_boston, n_pilot=5000)
print(f"Tuned refresh rate: {tune_info['lambda_r_old']:.3f} -> "
      f"{tune_info['lambda_r_new']:.3f} (floor active: {tune_info['floor_active']})")

sampler_boston_sticky = StickyAutomaticBoomerangSampler(
    grad_target=target_1d.grad_target,
    D=target_1d.D,
    thinning="pli",
    refresh_rate=1.0,
    kappa=kappa_from_weights
)

# Reference matches the prior — principled and needs no tuning
sampler_boston_sticky.preprocess(
    x_ref=target_1d.x_ref,
    Sigma_inv=target_1d.Sigma_inv
)

tune_info_sticky = tune_refresh_rate(sampler_boston_sticky, n_pilot=5000)
print(f"Tuned refresh rate: {tune_info_sticky['lambda_r_old']:.3f} -> "
      f"{tune_info_sticky['lambda_r_new']:.3f} (floor active: {tune_info_sticky['floor_active']})")


In [ ]:
from sazz.utils.bnn_utils import make_kappa_vector_bnn
from sazz.utils.warmup import warmup, tune_refresh_rate
T_MAX_ZZ = 0.1
GAMMA_ZZ = 0.01

sampler_boston_zz = AutomaticZigZagSampler(
    grad_target=target_1d.grad_target, 
    D=target_1d.D,
    t_max=T_MAX_ZZ, 
    gamma=GAMMA_ZZ, 
    thinning="pli",
)

sampler_boston_sticky_zz = StickyAutomaticZigZagSampler(
    grad_target=target_1d.grad_target, 
    D=target_1d.D,
    t_max=T_MAX_ZZ, 
    gamma=GAMMA_ZZ, 
    thinning="pli",
    kappa=kappa_from_weights,
)


In [ ]:
# --- Sample ---
N_SKELETON=10_000
result_boston = sampler_boston.sample(N=N_SKELETON, diagnostics=True)
result_boston_sticky = sampler_boston_sticky.sample(N=N_SKELETON, diagnostics=True)

result_boston_zz = sampler_boston_zz.sample(N=N_SKELETON, diagnostics=True, x0=target_1d.x_ref)
result_boston_sticky_zz = sampler_boston_sticky_zz.sample(N=N_SKELETON, diagnostics=True, x0=target_1d.x_ref)

In [ ]:
import pymc as pm
import numpy as np

# --- Match the BNN priors and architecture ---
layer_sizes = [1, 16, 8, 1]
activation_np = np.tanh
prior_std_weight = 1.0
prior_std_bias = 1.0
fan_in_scaling = True

X_train_np = datasets_1d['multiscale']['X_train'].numpy()
y_train_np = datasets_1d['multiscale']['y_train'].numpy()
noise_std_standardised = noise_std / datasets_1d['multiscale']['y_std']  # same as used in the Boomerang

# Per-layer prior stds (He-style if fan_in_scaling)
def prior_std(n_in):
    return prior_std_weight / np.sqrt(n_in) if fan_in_scaling else prior_std_weight

with pm.Model() as bnn_model:
    # Layer 0: input (1) -> hidden (10)
    W0 = pm.Normal('W0', mu=0.0, sigma=prior_std(layer_sizes[0]),
                   shape=(layer_sizes[1], layer_sizes[0]))
    b0 = pm.Normal('b0', mu=0.0, sigma=prior_std_bias,
                   shape=(layer_sizes[1],))

    # Layer 1: hidden (10) -> output (1)
    W1 = pm.Normal('W1', mu=0.0, sigma=prior_std(layer_sizes[1]),
                   shape=(layer_sizes[2], layer_sizes[1]))
    b1 = pm.Normal('b1', mu=0.0, sigma=prior_std_bias,
                   shape=(layer_sizes[2],))
    
    # Layer 2: hidden (10) -> output (1)
    W2 = pm.Normal('W2', mu=0.0, sigma=prior_std(layer_sizes[2]),
                   shape=(layer_sizes[3], layer_sizes[2]))
    b2 = pm.Normal('b2', mu=0.0, sigma=prior_std_bias,
                   shape=(layer_sizes[3],))

    # Forward pass — matches your BNNLikelihood.predict
    h1 = pm.math.tanh(X_train_np @ W0.T + b0)
    h2 = pm.math.tanh(h1 @ W1.T + b1)
    mu = (h2 @ W2.T + b2).squeeze(-1)

    # Likelihood
    pm.Normal('y', mu=mu, sigma=noise_std_standardised, observed=y_train_np)

    nuts_trace = pm.sample(
        draws=2000, tune=1000, chains=2,
        target_accept=0.9, progressbar=True, random_seed=0,
    )

# --- Flatten to match the Boomerang parameterisation ---
# Convention: [W0 flat, b0, W1 flat, b1]
W0_s = nuts_trace.posterior['W0'].values.reshape(-1, layer_sizes[1] * layer_sizes[0])
b0_s = nuts_trace.posterior['b0'].values.reshape(-1, layer_sizes[1])
W1_s = nuts_trace.posterior['W1'].values.reshape(-1, layer_sizes[2] * layer_sizes[1])
b1_s = nuts_trace.posterior['b1'].values.reshape(-1, layer_sizes[2])
W2_s = nuts_trace.posterior['W2'].values.reshape(-1, layer_sizes[3] * layer_sizes[2])
b2_s = nuts_trace.posterior['b2'].values.reshape(-1, layer_sizes[3])

samples_nuts = np.concatenate([W0_s, b0_s, W1_s, b1_s, W2_s, b2_s], axis=1)
print(f"NUTS: {samples_nuts.shape[0]} samples × {samples_nuts.shape[1]} parameters")
assert samples_nuts.shape[1] == target_1d.D, \
    f"Shape mismatch: NUTS gave {samples_nuts.shape[1]}, target.D={target_1d.D}"

In [ ]:

# ============================================================================
# Evaluate: RMSE, log-lik
# ============================================================================
from sazz.utils.sampling import resample_boomerang_path, resample_boomerang_path_sticky, resample_zigzag_path, resample_zigzag_path_sticky
from sazz.models.bnn_torch import predict_regression

BURNIN_FRAC = 0.1
N_RESAMPLE = 50_000

@torch.no_grad()
def predict_regression_raw(samples, X_test, target):
    """Like predict_regression, but returns the raw [n_samples, n_test] tensor
    so we can compute mean, std, median, quantiles, etc. ourselves."""
    likelihood = target.meta["model"].likelihood
    X_test = X_test.to(dtype=likelihood.X.dtype, device=likelihood.X.device)
    preds = torch.stack([
        likelihood.predict(beta, X_test).squeeze(-1) for beta in samples
    ])
    return preds  # shape [n_samples, n_test_points]


def resample_and_predict(result, target, model="boom"):
    if model == "boom":
        rsm = resample_boomerang_path
        samples_np = rsm(
            result["positions"].cpu().numpy(),
            result["velocities"].cpu().numpy(),
            result["times"].cpu().numpy(),
            target.x_ref.cpu().numpy(),
            N_resample=N_RESAMPLE,
            burnin_frac=BURNIN_FRAC,
        )
    elif model == "boom_sticky":
        rsm = resample_boomerang_path_sticky
        samples_np = rsm(
            result["positions"].cpu().numpy(),
            result["velocities"].cpu().numpy(),
            result["times"].cpu().numpy(),
            target.x_ref.cpu().numpy(),
            N_resample=N_RESAMPLE,
            burnin_frac=BURNIN_FRAC,
        )
    elif model == "zz":
        rsm = resample_zigzag_path
        samples_np = rsm(
            result["positions"].cpu().numpy(),
            result["velocities"].cpu().numpy(),
            result["times"].cpu().numpy(),
            #target.x_ref.cpu().numpy(),
            N_resample=N_RESAMPLE,
            burnin_frac=BURNIN_FRAC,
        )
    elif model == "zz_sticky":
        rsm = resample_zigzag_path_sticky
        samples_np = rsm(
            result["positions"].cpu().numpy(),
            result["velocities"].cpu().numpy(),
            result["times"].cpu().numpy(),
            #target.x_ref.cpu().numpy(),
            N_resample=N_RESAMPLE,
            burnin_frac=BURNIN_FRAC,
        )
    else:
        print("No model specified!")
        

    samples = torch.tensor(samples_np, dtype=torch.float64)
    preds = predict_regression_raw(samples, X_test_1d, target)
    mean = preds.mean(0)
    std = preds.std(0)
    median = preds.median(0).values
    return samples, mean, std, median, preds


def metrics(mean_pred, std_pred, y_test, noise_std):
    rmse = ((mean_pred - y_test) ** 2).mean().sqrt()
    total_std = (std_pred ** 2 + noise_std ** 2).sqrt()
    log_lik = (
        -0.5 * ((y_test - mean_pred) / total_std) ** 2
        - total_std.log()
        - 0.5 * torch.log(torch.tensor(2 * torch.pi))
    ).mean()
    return rmse, log_lik


X_test_1d = datasets_1d['multiscale']['X_test']
y_test_1d = datasets_1d['multiscale']['y_test']
noise_std = target_1d.meta["model"].likelihood.noise_std

samples_b,  mean_b,  std_b,  median_b,  preds_b  = resample_and_predict(result_boston, target_1d)
samples_bs, mean_bs, std_bs, median_bs, preds_bs = resample_and_predict(result_boston_sticky, target_1d, model="boom_sticky")
samples_z, mean_z, std_z, median_z, preds_z = resample_and_predict(result_boston_sticky, target_1d, model="zz")
samples_zs, mean_zs, std_zs, median_zs, preds_zs = resample_and_predict(result_boston_sticky, target_1d, model="zz_sticky")

samples_t_nuts = torch.tensor(samples_nuts, dtype=torch.float64)
preds_nuts = predict_regression_raw(samples_t_nuts, X_test_1d, target_1d)
mean_nuts = preds_nuts.mean(0)
std_nuts = preds_nuts.std(0)
median_nuts = preds_nuts.median(0).values

likelihood = target_1d.meta["model"].likelihood
X_test = X_test_1d.to(dtype=likelihood.X.dtype, device=likelihood.X.device)
preds_adam = likelihood.predict(target_1d.x_ref, X_test)


rmse_b,  ll_b  = metrics(mean_b,  std_b,  y_test_1d, noise_std)
rmse_bs, ll_bs = metrics(mean_bs, std_bs, y_test_1d, noise_std)
rmse_z,  ll_z  = metrics(mean_z,  std_z,  y_test_1d, noise_std)
rmse_zs, ll_zs = metrics(mean_zs, std_zs, y_test_1d, noise_std)
rmse_nuts, ll_nuts = metrics(mean_nuts, std_nuts, y_test_1d, noise_std)
rmse_adam = ((preds_adam - y_test_1d) ** 2).mean().sqrt()

print(f"{'Method':<20}{'RMSE':>10}{'LogLik':>10}{'Pred std':>12}")
print("-" * 52)
print(f"{'NUTS':<20}{rmse_nuts:>10.4f}{ll_nuts:>10.4f}{std_nuts.mean():>12.4f}")
print(f"{'Boomerang':<20}{rmse_b:>10.4f}{ll_b:>10.4f}{std_b.mean():>12.4f}")
print(f"{'Sticky Boomerang':<20}{rmse_bs:>10.4f}{ll_bs:>10.4f}{std_bs.mean():>12.4f}")
print(f"{'ZigZag':<20}{rmse_z:>10.4f}{ll_z:>10.4f}{std_z.mean():>12.4f}")
print(f"{'Sticky ZigZag':<20}{rmse_zs:>10.4f}{ll_zs:>10.4f}{std_zs.mean():>12.4f}")
print(f"{'Adam':<20}{rmse_adam:>10.4f}")

In [ ]:
# ============================================================================
# Plot: posterior predictive vs. true function (2x3 grid)
# Top row: predictive (epistemic + aleatoric)
# Bottom row: epistemic uncertainty alone
# ============================================================================
import matplotlib.pyplot as plt

data = datasets_1d['multiscale']
x_test_raw = data['x_test_raw']
y_test_raw = data['y_test_raw']
x_train_raw = data['x_train_raw']
y_train_raw = data['y_train_raw']
y_mean_s, y_std_s = data['y_mean'], data['y_std']

def to_original_scale(mean_s, std_s):
    """Undo standardisation: y = y_s * y_std + y_mean, std likewise."""
    return mean_s * y_std_s + y_mean_s, std_s * y_std_s

# Build total predictive std (epistemic + observation noise, in original units)
noise_std_orig = noise_std * y_std_s

mean_b_orig,  std_b_orig  = to_original_scale(mean_b,  std_b)
mean_bs_orig, std_bs_orig = to_original_scale(mean_bs, std_bs)
mean_z_orig,  std_z_orig  = to_original_scale(mean_z,  std_z)
mean_zs_orig, std_zs_orig = to_original_scale(mean_zs, std_zs)
mean_nuts_orig, std_nuts_orig = to_original_scale(mean_nuts, std_nuts)

# Upstream: compute medians alongside means (per test point, across posterior samples)
# Assuming you have predictive samples shaped [n_post_samples, n_test_points]
median_b    = preds_b.median(dim=0).values
median_bs   = preds_bs.median(dim=0).values
median_z    = preds_z.median(dim=0).values
median_zs   = preds_zs.median(dim=0).values
median_nuts = preds_nuts.median(dim=0).values

median_b_orig,    _ = to_original_scale(median_b,    torch.zeros_like(median_b))
median_bs_orig,   _ = to_original_scale(median_bs,   torch.zeros_like(median_bs))
median_z_orig,    _ = to_original_scale(median_z,    torch.zeros_like(median_z))
median_zs_orig,   _ = to_original_scale(median_zs,   torch.zeros_like(median_zs))
median_nuts_orig, _ = to_original_scale(median_nuts, torch.zeros_like(median_nuts))

total_b    = (std_b_orig    ** 2 + noise_std_orig ** 2).sqrt()
total_bs   = (std_bs_orig   ** 2 + noise_std_orig ** 2).sqrt()
total_z    = (std_z_orig    ** 2 + noise_std_orig ** 2).sqrt()
total_zs   = (std_zs_orig   ** 2 + noise_std_orig ** 2).sqrt()
total_nuts = (std_nuts_orig ** 2 + noise_std_orig ** 2).sqrt()

# 2x3 grid: predictive on top, epistemic on bottom
fig, axes = plt.subplots(2, 5, figsize=(16, 9), sharex=True)

methods = [
    ("NUTS",             mean_nuts_orig.numpy(), median_nuts_orig.numpy(), std_nuts_orig.numpy(), total_nuts.numpy()),
    ("Boomerang",        mean_b_orig.numpy(),    median_b_orig.numpy(),    std_b_orig.numpy(),    total_b.numpy()),
    ("Sticky Boomerang", mean_bs_orig.numpy(),   median_bs_orig.numpy(),   std_bs_orig.numpy(),   total_bs.numpy()),
    ("ZigZag",           mean_z_orig.numpy(),    median_z_orig.numpy(),    std_z_orig.numpy(),    total_z.numpy()),
    ("Sticky ZigZag",    mean_zs_orig.numpy(),   median_zs_orig.numpy(),   std_zs_orig.numpy(),   total_zs.numpy()),
]

# Top row: predictive (epistemic + aleatoric)
for col, (title, mean, median, _, total) in enumerate(methods):
    ax = axes[0, col]
    ax.plot(x_test_raw, y_test_raw, 'k-', lw=1.5, label='true function')
    ax.plot(x_train_raw, y_train_raw, 'k.', ms=4, alpha=0.6, label='training data')
    ax.plot(x_test_raw, mean, 'C0-', lw=2, label='posterior mean')
    ax.plot(x_test_raw, median, 'C3--', lw=1.5, label='posterior median')
    ax.fill_between(x_test_raw, mean - 2 * total, mean + 2 * total,
                    color='C0', alpha=0.25, label=r'predictive $\pm 2\sigma$')
    ax.set_title(title)
    if col == 0:
        ax.set_ylabel('y')
    ax.legend(loc='upper left', fontsize=9)

# Share y-axis across the top row only
y_min = min(ax.get_ylim()[0] for ax in axes[0, :])
y_max = max(ax.get_ylim()[1] for ax in axes[0, :])
for ax in axes[0, :]:
    ax.set_ylim(y_min, y_max)

# Bottom row: epistemic uncertainty alone
for col, (title, _, _, epistemic, _) in enumerate(methods):
    ax = axes[1, col]
    ax.plot(x_test_raw, epistemic, 'C2-', lw=2, label='epistemic std')
    ax.axhline(noise_std_orig, color='gray', linestyle='--', label='noise std')
    for x_tr in x_train_raw:
        ax.axvline(x_tr, color='k', alpha=0.05)
    ax.set_xlabel('x')
    if col == 0:
        ax.set_ylabel('std')
        ax.legend(loc='upper left', fontsize=9)

# Share y-axis across the bottom row only
y_min = min(ax.get_ylim()[0] for ax in axes[1, :])
y_max = max(ax.get_ylim()[1] for ax in axes[1, :])
for ax in axes[1, :]:
    ax.set_ylim(y_min, y_max)

plt.tight_layout()
plt.show()